In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.mimic_bronze"
SILVER = f"{CATALOG}.mimic_silver"

from pyspark.sql import functions as F
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

def save(df, table):
    df.write.mode("overwrite").format("delta").saveAsTable(f"{SILVER}.{table}")
    print(f"{SILVER}.{table}: {spark.table(f'{SILVER}.{table}').count():,} rows")

In [0]:
# types and icd tables
# procedures_icd: fix data types, keep icd_code as string
pi = spark.table(f"{BRONZE}.procedures_icd")
pi = (pi.withColumn("subject_id", F.col("subject_id").cast("int"))
        .withColumn("hadm_id", F.col("hadm_id").cast("int"))
        .withColumn("seq_num", F.col("seq_num").cast("int"))
        .withColumn("icd_version", F.col("icd_version").cast("int"))
        .withColumn("chartdate", F.col("chartdate").cast("date"))
        # add a readable version of ICD-9 codes: "0066" becomes "00.66"
        .withColumn("icd_code_display",
            F.when((F.col("icd_version") == 9) & (F.length("icd_code") > 2),
                   F.concat(F.substring("icd_code", 1, 2), F.lit("."), F.expr("substring(icd_code, 3)")))
             .otherwise(F.col("icd_code"))))
save(pi, "procedures_icd")

# d_icd_procedures: fix types only, codes stay as string
dp = spark.table(f"{BRONZE}.d_icd_procedures") \
        .withColumn("icd_version", F.col("icd_version").cast("int"))
save(dp, "d_icd_procedures")

In [0]:
# procedureevents: proper timestamps and how long each procedure took
pe = spark.table(f"{BRONZE}.procedureevents")
pe = (pe.withColumn("starttime", F.col("starttime").cast("timestamp"))
        .withColumn("endtime", F.col("endtime").cast("timestamp"))
        .withColumn("duration_minutes",
            F.round((F.unix_timestamp("endtime") - F.unix_timestamp("starttime")) / 60, 1)))
save(pe, "procedureevents")

# inputevents: merge the two spellings of mEq into one + duration + flag zero amounts
ie = spark.table(f"{BRONZE}.inputevents")
ie = (ie.withColumn("amountuom", F.when(F.col("amountuom") == "mEq.", "mEq").otherwise(F.col("amountuom")))
        .withColumn("starttime", F.col("starttime").cast("timestamp"))
        .withColumn("endtime", F.col("endtime").cast("timestamp"))
        .withColumn("duration_minutes",
            F.round((F.unix_timestamp("endtime") - F.unix_timestamp("starttime")) / 60, 1))
        .withColumn("is_zero_amount", (F.col("amount") == 0)))
save(ie, "inputevents")

# ingredientevents: same treatment as inputevents
ig = spark.table(f"{BRONZE}.ingredientevents")
ig = (ig.withColumn("starttime", F.col("starttime").cast("timestamp"))
        .withColumn("endtime", F.col("endtime").cast("timestamp"))
        .withColumn("is_zero_amount", (F.col("amount") == 0)))
save(ig, "ingredientevents")

# outputevents: proper timestamp & flag zero values (units are already uniform)
oe = spark.table(f"{BRONZE}.outputevents")
oe = (oe.withColumn("charttime", F.col("charttime").cast("timestamp"))
        .withColumn("is_zero_value", (F.col("value") == 0)))
save(oe, "outputevents")

# check: only one spelling of mEq left
spark.table(f"{SILVER}.inputevents").filter("amountuom LIKE 'mEq%'").groupBy("amountuom").count().show()

In [0]:
# compare row counts between bronze and silver for every table
rows = []
for t in ["procedures_icd", "d_icd_procedures", "procedureevents",
          "inputevents", "ingredientevents", "outputevents"]:
    b = spark.table(f"{BRONZE}.{t}").count()
    s = spark.table(f"{SILVER}.{t}").count()
    rows.append((t, b, s, round(s / b * 100, 2)))

dq = spark.createDataFrame(rows, ["table", "bronze_rows", "silver_rows", "retained_pct"])
dq.write.mode("overwrite").format("delta").saveAsTable(f"{SILVER}.dq_metrics_hansen")
dq.show()
# no rows are dropped in this domain (we flag instead of filter), so retained_pct should be 100